### Step A: Initialize and Create the Table

First, ensure the extension is loaded and define a table. Let's assume we are storing **1536-dimensional face embeddings** for an enterprise security system (where $L_2$ is the ideal metric).

```sql
-- Enable the pgvector extension
CREATE EXTENSION IF NOT EXISTS vector;

-- Create a table for face recognition profiles
CREATE TABLE user_biometrics (
    id SERIAL PRIMARY KEY,
    user_name VARCHAR(100),
    face_embedding vector(1536) -- 1536-dimensional vector
);

```

### Step B: Create the HNSW Index using $L_2$

To tell `pgvector` to build an HNSW graph optimized for Euclidean distance, you use the **`vector_l2_ops`** operator class.

```sql
CREATE INDEX ON user_biometrics 
USING hnsw (face_embedding vector_l2_ops) -- <--- Instructs HNSW to use L2 math
WITH (m = 16, ef_construction = 64);

```

> **Architectural Note on HNSW Parameters:**
> * `m = 16`: The maximum number of bidirectional connection links created for each new node in the graph layers.
> * `ef_construction = 64`: Specifies the size of the dynamic candidate list evaluated during index building. Higher numbers mean better recall but slower index generation.
> 
> 

### Step C: Querying the HNSW Index using $L_2$

To perform a similarity search and utilize the HNSW index you just created, you must use the $L_2$ operator **`<->`** (which calculates Euclidean distance).

```sql
-- Find the top 5 closest matching faces to an incoming scan
SELECT id, user_name, face_embedding <-> '[0.012, -0.043, ..., 0.122]' AS distance
FROM user_biometrics
ORDER BY face_embedding <-> '[0.012, -0.043, ..., 0.122]'
LIMIT 5;

```

> **Crucial Rule:** The operator in your `ORDER BY` clause (**`<->`**) must match the operator class used to build the index (**`vector_l2_ops`**). If you accidentally use the Cosine operator (`<=>`) or Dot Product operator (`<#>`), PostgreSQL will bypass your HNSW index and perform an expensive sequential scan across the whole database.

---

## 2. A Comparative Reference for `pgvector` Metrics

When you write or review migration files, you can use this quick syntax reference to ensure you are binding the right math to the right index type:

| Metric | Index Operator Class | Query Distance Operator | Perfect Use Case |
| --- | --- | --- | --- |
| **$L_2$ Distance** | `vector_l2_ops` | **`<->`** (Euclidean Distance) | Computer Vision, Audio Recognition, Fixed Coordinate Models |
| **Cosine Distance** | `vector_cos_ops` | **`<=>`** (Cosine Distance) | Text Retrieval, Variable-length document RAG |
| **Dot Product** | `vector_ip_ops` | **`<#>`** (Negative Inner Product)* | RecSys (Popularity-weighted), Pre-normalized vectors |

**Note: `pgvector` uses negative inner product (`<#>`) because PostgreSQL indexes look for the smallest value during an `ORDER BY` clause, and a smaller negative number corresponds to a larger, more similar raw dot product.*

# Complete Enterprise Cheat Sheet for `pgvector` in PostgreSQL

## What is `pgvector`?

`pgvector` is an extension for PostgreSQL that adds **vector data type support** and enables:

* Vector similarity search
* Semantic search
* RAG systems
* AI embeddings storage
* Hybrid search
* Recommendation systems
* Nearest Neighbor Search (ANN)
* Image/Text similarity applications

It allows PostgreSQL to act as a **Vector Database** while still supporting:

* ACID transactions
* SQL joins
* Indexing
* Replication
* Backup
* Enterprise security
* Structured + unstructured data together

---

# Why Enterprises Use pgvector

## Main Enterprise Advantages

| Feature              | Benefit                            |
| -------------------- | ---------------------------------- |
| PostgreSQL ecosystem | No separate vector DB needed       |
| SQL support          | Easier integration                 |
| ACID compliance      | Enterprise reliability             |
| Hybrid search        | Combine metadata + vector search   |
| Scalability          | Works with partitioning & replicas |
| Cost effective       | No dedicated vector DB infra       |
| Existing tooling     | pgAdmin, backups, monitoring       |
| Security             | RBAC, row-level security           |

---

# Common Enterprise Use Cases

---

## 1. RAG (Retrieval Augmented Generation)

Store embeddings of:

* PDFs
* Policies
* SOPs
* Knowledge bases
* Emails
* Jira tickets
* Confluence pages

### Flow

```text
User Query
   ↓
Generate Embedding
   ↓
pgvector Similarity Search
   ↓
Top-k Relevant Chunks
   ↓
LLM Context
   ↓
Generated Answer
```

---

## 2. Semantic Search

Traditional search:

```sql
WHERE title LIKE '%database%'
```

Semantic search:

```text
"database scaling"
≈
"horizontal partitioning"
≈
"distributed architecture"
```

---

## 3. Recommendation Engines

Examples:

* Product recommendations
* Video recommendations
* Job recommendations
* Music recommendation

---

## 4. Fraud Detection

Compare embeddings of:

* User behavior
* Transaction patterns
* Device fingerprints

---

## 5. Image Similarity Search

Store image embeddings from:

* CLIP
* OpenAI
* HuggingFace

Find visually similar images.

---

## 6. Hybrid Search

Combine:

* BM25 keyword search
* Metadata filtering
* Semantic vector search

Enterprise-grade RAG systems heavily use this.

---

# Installation

## Enable Extension

```sql
CREATE EXTENSION IF NOT EXISTS vector;
```

Check installed extensions:

```sql
SELECT * FROM pg_extension;
```

---

# Vector Data Types

## Main Vector Types

| Type           | Description    |
| -------------- | -------------- |
| `vector(n)`    | Dense vector   |
| `halfvec(n)`   | Half precision |
| `bit(n)`       | Binary vectors |
| `sparsevec(n)` | Sparse vectors |

---

# Creating Tables

---

## Basic Table

```sql
CREATE TABLE documents (
    id BIGSERIAL PRIMARY KEY,
    content TEXT,
    embedding VECTOR(1536)
);
```

---

## Enterprise Grade Table

```sql
CREATE TABLE knowledge_base (
    id BIGSERIAL PRIMARY KEY,
    
    document_id UUID,
    chunk_id INT,
    
    title TEXT,
    content TEXT,
    
    source_type VARCHAR(50),
    source_path TEXT,
    
    metadata JSONB,
    
    embedding VECTOR(1536),
    
    created_at TIMESTAMP DEFAULT NOW(),
    updated_at TIMESTAMP DEFAULT NOW()
);
```

---

# Why These Enterprise Columns Matter

| Column        | Purpose                  |
| ------------- | ------------------------ |
| `document_id` | Parent document tracking |
| `chunk_id`    | Chunk ordering           |
| `metadata`    | Dynamic filters          |
| `source_type` | PDF, DOCX, Email         |
| `created_at`  | Auditing                 |
| `updated_at`  | Version tracking         |

---

# Insert Data

---

## Insert Single Vector

```sql
INSERT INTO documents (content, embedding)
VALUES (
    'PostgreSQL vector database',
    '[0.12, 0.45, 0.87]'
);
```

---

## Insert Enterprise Metadata

```sql
INSERT INTO knowledge_base (
    document_id,
    chunk_id,
    title,
    content,
    source_type,
    metadata,
    embedding
)
VALUES (
    gen_random_uuid(),
    1,
    'Vector Databases',
    'pgvector supports vector similarity search',
    'PDF',
    '{"department":"AI","access":"internal"}',
    '[0.11,0.22,0.33]'
);
```

---

# Vector Distance Metrics

Distance metrics are critical.

---

# 1. L2 Distance (Euclidean)

## Operator

```sql
<-> 
```

## Formula

d(a,b)=\sqrt{\sum_{i=1}^{n}(a_i-b_i)^2}

## Query

```sql
SELECT *
FROM documents
ORDER BY embedding <-> '[0.1,0.2,0.3]'
LIMIT 5;
```

## Best For

* General embeddings
* Geometric distance
* Clustering

---

# 2. Cosine Distance

## Operator

```sql
<=> 
```

## Formula

\cos(\theta)=\frac{A\cdot B}{||A||\ ||B||}

## Query

```sql
SELECT *
FROM documents
ORDER BY embedding <=> '[0.1,0.2,0.3]'
LIMIT 5;
```

## Most Common in AI

Used for:

* OpenAI embeddings
* Sentence transformers
* BERT embeddings

---

# 3. Inner Product

## Operator

```sql
<#>
```

## Formula

A\cdot B=\sum_{i=1}^{n}A_iB_i

## Query

```sql
SELECT *
FROM documents
ORDER BY embedding <#> '[0.1,0.2,0.3]'
LIMIT 5;
```

## Best For

* Recommendation systems
* Maximum similarity scoring

---

# Which Metric Should You Use?

| Embedding Type            | Recommended Metric |
| ------------------------- | ------------------ |
| OpenAI                    | Cosine             |
| Sentence Transformers     | Cosine             |
| CLIP                      | Cosine             |
| Recommendation embeddings | Inner Product      |
| Spatial vectors           | Euclidean          |

---

# Similarity vs Distance

| Concept    | Meaning           |
| ---------- | ----------------- |
| Distance   | Smaller is better |
| Similarity | Larger is better  |

---

# Convert Cosine Distance to Similarity

```sql
SELECT
    1 - (embedding <=> '[0.1,0.2,0.3]') AS similarity
FROM documents;
```

---

# Top-K Similarity Search

```sql
SELECT
    id,
    content,
    1 - (embedding <=> '[0.1,0.2,0.3]') AS similarity
FROM documents
ORDER BY embedding <=> '[0.1,0.2,0.3]'
LIMIT 10;
```

---

# Threshold Filtering

## Only Highly Similar Results

```sql
SELECT *
FROM documents
WHERE 1 - (embedding <=> '[0.1,0.2,0.3]') > 0.80
ORDER BY embedding <=> '[0.1,0.2,0.3]';
```

---

# Hybrid Search (Enterprise Important)

Combine:

* Metadata filters
* Keyword search
* Vector search

---

## Example

```sql
SELECT
    id,
    title,
    content
FROM knowledge_base
WHERE metadata->>'department' = 'AI'
AND source_type = 'PDF'
ORDER BY embedding <=> '[0.1,0.2,0.3]'
LIMIT 5;
```

---

# Full Text + Vector Search

```sql
SELECT *
FROM knowledge_base
WHERE to_tsvector(content) @@ plainto_tsquery('postgresql')
ORDER BY embedding <=> '[0.1,0.2,0.3]'
LIMIT 5;
```

---

# Important pgvector Index Types

Without indexes:

```text
O(N) scan
```

Very slow at enterprise scale.

---

# 1. IVFFLAT Index

## Create Index

```sql
CREATE INDEX idx_embedding_ivfflat
ON documents
USING ivfflat (embedding vector_cosine_ops)
WITH (lists = 100);
```

---

## Important Parameter: `lists`

| Value     | Effect        |
| --------- | ------------- |
| Small     | Faster build  |
| Large     | Better recall |
| Too large | More memory   |

---

## Enterprise Recommendation

| Rows | Suggested Lists |
| ---- | --------------- |
| 100K | 100             |
| 1M   | 1000            |
| 10M  | 5000            |

---

# Query Parameter: `probes`

Controls search accuracy.

```sql
SET ivfflat.probes = 10;
```

---

## Probe Tradeoff

| Probes | Result        |
| ------ | ------------- |
| Low    | Faster        |
| High   | Better recall |

---

# 2. HNSW Index (Most Important)

Modern enterprise choice.

---

## Create HNSW Index

```sql
CREATE INDEX idx_embedding_hnsw
ON documents
USING hnsw (embedding vector_cosine_ops);
```

---

# HNSW Parameters

---

## 1. `m`

Controls graph connectivity.

```sql
WITH (m = 16)
```

| Higher m      | Effect        |
| ------------- | ------------- |
| Better recall | More memory   |
| Slower build  | Better search |

---

## 2. `ef_construction`

Controls indexing quality.

```sql
WITH (
    m = 16,
    ef_construction = 64
)
```

Higher value:

* Better accuracy
* Slower indexing

---

## 3. `ef_search`

Controls search recall.

```sql
SET hnsw.ef_search = 100;
```

Higher:

* Better recall
* Slower query

---

# Enterprise HNSW Example

```sql
CREATE INDEX idx_hnsw_embedding
ON knowledge_base
USING hnsw (embedding vector_cosine_ops)
WITH (
    m = 16,
    ef_construction = 64
);
```

---

# Index Operator Classes

| Operator Class      | Distance      |
| ------------------- | ------------- |
| `vector_l2_ops`     | Euclidean     |
| `vector_cosine_ops` | Cosine        |
| `vector_ip_ops`     | Inner product |

---

# Enterprise Search Query

```sql
SELECT
    id,
    title,
    source_type,
    metadata,
    1 - (embedding <=> '[0.1,0.2,0.3]') AS similarity
FROM knowledge_base
WHERE metadata->>'access' = 'internal'
ORDER BY embedding <=> '[0.1,0.2,0.3]'
LIMIT 10;
```

---

# Chunking Strategy (Very Important)

## Recommended Chunk Sizes

| Content Type | Chunk Size      |
| ------------ | --------------- |
| PDFs         | 500–1000 tokens |
| Code         | 200–400 tokens  |
| Emails       | 300–500 tokens  |

---

# Metadata Best Practices

Store:

```json
{
  "department": "finance",
  "country": "india",
  "classification": "internal",
  "document_type": "policy"
}
```

Enables:

* RBAC filtering
* Department filtering
* Compliance filtering

---

# Common Enterprise Keywords

| Keyword          | Meaning                      |
| ---------------- | ---------------------------- |
| ANN              | Approximate Nearest Neighbor |
| Recall           | Retrieval accuracy           |
| Precision        | Result correctness           |
| Embedding        | Numeric representation       |
| Vector Dimension | Embedding size               |
| Top-K            | Number of retrieved results  |
| Hybrid Search    | Keyword + vector             |
| Reranking        | Second-stage ranking         |
| Recall@K         | Accuracy metric              |
| Latency          | Query response time          |
| Throughput       | Queries/sec                  |
| Semantic Search  | Meaning-based search         |

---

# RAG Architecture with pgvector

```text
PDF/DOCX
   ↓
Chunking
   ↓
Embedding Model
   ↓
pgvector Storage
   ↓
Similarity Search
   ↓
Top-k Chunks
   ↓
LLM Prompt
   ↓
Answer
```

---

# Recommended Embedding Dimensions

| Model                         | Dimensions |
| ----------------------------- | ---------- |
| OpenAI text-embedding-3-small | 1536       |
| OpenAI text-embedding-3-large | 3072       |
| BGE Small                     | 384        |
| MiniLM                        | 384        |
| E5 Base                       | 768        |

---

# Partitioning for Enterprise Scale

## Example

```sql
CREATE TABLE kb_2026 PARTITION OF knowledge_base
FOR VALUES FROM ('2026-01-01') TO ('2027-01-01');
```

Use when:

* Billions of vectors
* Multi-tenant systems
* Time-based partitioning

---

# Multi-Tenant Architecture

```sql
CREATE TABLE tenant_documents (
    tenant_id UUID,
    document_id UUID,
    embedding VECTOR(1536),
    content TEXT
);
```

---

# Enterprise Security Filtering

```sql
SELECT *
FROM knowledge_base
WHERE metadata->>'role' = 'manager'
ORDER BY embedding <=> '[0.1,0.2,0.3]'
LIMIT 5;
```

---

# Common Enterprise Optimization Techniques

| Optimization               | Purpose          |
| -------------------------- | ---------------- |
| HNSW                       | Faster ANN       |
| Partitioning               | Scale            |
| Connection pooling         | Throughput       |
| Batch inserts              | Faster ingestion |
| JSONB indexing             | Metadata speed   |
| Async embedding generation | Better ingestion |
| Hybrid reranking           | Better accuracy  |

---

# Batch Insert Example

```sql
INSERT INTO documents (content, embedding)
VALUES
('doc1', '[0.1,0.2,0.3]'),
('doc2', '[0.4,0.5,0.6]'),
('doc3', '[0.7,0.8,0.9]');
```

---

# Update Embeddings

```sql
UPDATE documents
SET embedding = '[0.11,0.22,0.33]'
WHERE id = 1;
```

---

# Delete Vectors

```sql
DELETE FROM documents
WHERE id = 1;
```

---

# Performance Tuning Queries

---

## Analyze Table

```sql
ANALYZE documents;
```

---

## Check Query Plan

```sql
EXPLAIN ANALYZE
SELECT *
FROM documents
ORDER BY embedding <=> '[0.1,0.2,0.3]'
LIMIT 5;
```

---

# Enterprise Scale Considerations

---

## When pgvector Works Best

| Scenario                  | Good Fit  |
| ------------------------- | --------- |
| Existing PostgreSQL infra | Excellent |
| RAG systems               | Excellent |
| Medium-large scale        | Excellent |
| Hybrid SQL + vector       | Excellent |

---

## When Dedicated Vector DB May Be Better

| Scenario          | Better Choice         |
| ----------------- | --------------------- |
| 100B+ vectors     | Specialized vector DB |
| Ultra-low latency | Milvus/FAISS          |
| GPU ANN           | Dedicated systems     |

---

# features of pgvector

| Feature | pgvector |
| ----- |  ----- |
| SQL | Yes |
| Persistence | Yes |
| ACID | Yes |
| GPU | No |
| Metadata filtering | Excellent |
| Enterprise integration | Excellent |
| ANN performance | Good |
| Distributed support | Limited |

---

# Enterprise Stack Example

| Layer             | Technology             |
| ----------------- | ---------------------- |
| API               | FastAPI                |
| Embeddings        | OpenAI                 |
| Vector Store      | PostgreSQL + pgvector  |
| Cache             | Redis                  |
| Object Storage    | Amazon Web Services S3 |
| LLM Orchestration | LangChain              |
| Workflow          | LangGraph              |

---

# Common Interview Questions

---

## Why choose pgvector over dedicated vector DB?

Answer:

* Existing PostgreSQL ecosystem
* Easier operations
* Hybrid search
* ACID guarantees
* Lower infrastructure complexity

---

## Why use cosine similarity?

Because embedding magnitude is less important than directional similarity.

---

## Difference between IVFFLAT and HNSW?

| IVFFLAT      | HNSW          |
| ------------ | ------------- |
| Faster build | Better recall |
| Lower memory | Higher memory |
| Simpler      | More accurate |

---

# Production Best Practices

---

## DO

✅ Use HNSW for production
✅ Store metadata in JSONB
✅ Use hybrid search
✅ Use chunking
✅ Add security filters
✅ Monitor recall and latency
✅ Batch embedding generation

---

## DON'T

❌ Store huge raw documents in same row
❌ Use sequential scan at scale
❌ Ignore metadata filters
❌ Use incorrect embedding dimensions
❌ Forget ANALYZE/VACUUM

---

# Advanced Enterprise Query Example

```sql
SELECT
    id,
    title,
    content,
    metadata,
    1 - (embedding <=> '[0.1,0.2,0.3]') AS similarity
FROM knowledge_base
WHERE
    metadata->>'department' = 'finance'
    AND metadata->>'country' = 'india'
    AND created_at >= NOW() - INTERVAL '30 days'
ORDER BY embedding <=> '[0.1,0.2,0.3]'
LIMIT 20;
```

---

# Final Enterprise Takeaway

`pgvector` is best when you need:

* AI + SQL together
* RAG systems
* Enterprise metadata filtering
* Existing PostgreSQL infrastructure
* Lower operational complexity
* Hybrid semantic search

It is one of the most practical enterprise choices for:

* Internal GenAI platforms
* Enterprise search
* AI copilots
* Knowledge assistants
* Multi-tenant RAG systems


# Complete pgvector Keywords, Operators & Symbols Cheat Sheet

## (Ordered by Enterprise Usage & Popularity)

This cheat sheet focuses on:

* pgvector operators
* PostgreSQL keywords used with pgvector
* Vector query syntax
* JSONB operators
* ANN indexing keywords
* Enterprise filtering patterns
* SQL constructs heavily used in RAG and GenAI systems

---

# Tier-1: Most Used & Most Important

These are used in almost every enterprise pgvector application.

---

# 1. `<=>` — Cosine Distance Operator

## Most Popular Operator in AI Systems

Used for:

* Semantic search
* RAG
* OpenAI embeddings
* Sentence transformers

---

## Syntax

```sql
embedding <=> query_vector
```

---

## Example

```sql
SELECT *
FROM documents
ORDER BY embedding <=> '[0.1,0.2,0.3]'
LIMIT 5;
```

---

## Meaning

Returns:

```text
Cosine Distance
```

Smaller value = More similar.

---

## Convert to Similarity

```sql
1 - (embedding <=> query_vector)
```

---

## Enterprise Usage

| Use Case          | Usage          |
| ----------------- | -------------- |
| RAG               | Extremely High |
| Semantic search   | Extremely High |
| AI copilots       | Extremely High |
| Chatbot retrieval | Extremely High |

---

# 2. `ORDER BY`

## Core Vector Search Keyword

Critical because pgvector search works using:

```text
Nearest Neighbor Ordering
```

---

## Example

```sql
ORDER BY embedding <=> '[0.1,0.2,0.3]'
```

---

## Why Important

Without `ORDER BY`, similarity search does not work properly.

---

# 3. `LIMIT`

## Top-K Retrieval

Used to fetch:

```text
Top nearest vectors
```

---

## Example

```sql
LIMIT 10
```

---

## Enterprise Importance

Controls:

* LLM context size
* Latency
* Recall quality
* Token usage

---

# 4. `VECTOR(n)`

## Vector Data Type

Defines embedding dimensions.

---

## Syntax

```sql
embedding VECTOR(1536)
```

---

## Meaning

```text
1536-dimensional embedding
```

---

## Common Dimensions

| Model        | Dimensions |
| ------------ | ---------- |
| OpenAI small | 1536       |
| OpenAI large | 3072       |
| BGE          | 384        |
| MiniLM       | 384        |

---

# 5. `USING hnsw`

## Most Popular ANN Index

Used for:

* Fast similarity search
* Production RAG systems

---

## Example

```sql
CREATE INDEX idx_embedding
ON documents
USING hnsw (embedding vector_cosine_ops);
```

---

## Why Important

Provides:

```text
Approximate Nearest Neighbor Search
```

instead of full table scan.

---

# 6. `vector_cosine_ops`

## Index Operator Class

Defines:

```text
Which distance metric index uses
```

---

## Example

```sql
USING hnsw (embedding vector_cosine_ops)
```

---

## Other Variants

| Operator Class      | Meaning       |
| ------------------- | ------------- |
| `vector_cosine_ops` | Cosine        |
| `vector_l2_ops`     | Euclidean     |
| `vector_ip_ops`     | Inner product |

---

# 7. `WHERE`

## Metadata Filtering

Most enterprise systems combine:

* Vector search
* Metadata filtering

---

## Example

```sql
WHERE metadata->>'department' = 'finance'
```

---

## Enterprise Importance

Critical for:

* RBAC
* Tenant isolation
* Security
* Compliance

---

# 8. `->>` — JSONB Text Extraction Operator

## Extremely Important Enterprise Operator

Extracts JSON value as text.

---

## Example

```sql
metadata->>'department'
```

---

## JSON Example

```json
{
  "department": "finance"
}
```

---

## Result

```text
finance
```

---

# 9. `CREATE EXTENSION vector`

## Enables pgvector

---

## Example

```sql
CREATE EXTENSION vector;
```

---

## Meaning

Adds vector support to PostgreSQL.

---

# 10. `CREATE INDEX`

## Performance Critical

Without indexes:

```text
Full table scan
```

---

## Example

```sql
CREATE INDEX idx_embedding
ON documents
USING hnsw (embedding vector_cosine_ops);
```

---

# Tier-2: Very Common Enterprise Keywords

---

# 11. `<->` — Euclidean Distance

## L2 Distance

---

## Formula

d(a,b)=\sqrt{\sum_{i=1}^{n}(a_i-b_i)^2}

---

## Example

```sql
ORDER BY embedding <-> '[0.1,0.2,0.3]'
```

---

## Usage

* Clustering
* Spatial embeddings
* Mathematical similarity

---

# 12. `<#>` — Inner Product

## Dot Product Similarity

---

## Formula

A\cdot B=\sum_{i=1}^{n}A_iB_i

---

## Example

```sql
ORDER BY embedding <#> '[0.1,0.2,0.3]'
```

---

## Usage

* Recommendation systems
* Ranking systems

---

# 13. `WITH (...)`

## Index Configuration Parameters

Used while creating ANN indexes.

---

## Example

```sql
WITH (
    m = 16,
    ef_construction = 64
)
```

---

# 14. `m`

## HNSW Graph Connectivity

---

## Example

```sql
WITH (m = 16)
```

---

## Meaning

Controls:

* Graph density
* Accuracy
* Memory usage

---

# 15. `ef_construction`

## HNSW Build Accuracy

---

## Example

```sql
WITH (
    ef_construction = 64
)
```

---

## Meaning

Higher value:

* Better recall
* Slower indexing

---

# 16. `SET hnsw.ef_search`

## Query Recall Tuning

---

## Example

```sql
SET hnsw.ef_search = 100;
```

---

## Meaning

Controls:

* Query accuracy
* Search breadth

---

# 17. `USING ivfflat`

## Older ANN Index

Still widely used.

---

## Example

```sql
USING ivfflat (embedding vector_cosine_ops)
```

---

# 18. `lists`

## IVFFLAT Clusters

---

## Example

```sql
WITH (lists = 100)
```

---

## Meaning

Higher lists:

* Better recall
* More memory

---

# 19. `SET ivfflat.probes`

## IVFFLAT Search Accuracy

---

## Example

```sql
SET ivfflat.probes = 10;
```

---

# 20. `JSONB`

## Enterprise Metadata Storage

---

## Example

```sql
metadata JSONB
```

---

## Used For

* RBAC
* Dynamic filters
* Multi-tenant metadata
* Compliance labels

---

# Tier-3: Frequently Used in Enterprise RAG

---

# 21. `to_tsvector()`

## Full Text Search Conversion

---

## Example

```sql
to_tsvector(content)
```

---

# 22. `plainto_tsquery()`

## Keyword Query Builder

---

## Example

```sql
plainto_tsquery('postgresql')
```

---

# 23. `@@`

## Full Text Match Operator

---

## Example

```sql
to_tsvector(content) @@ plainto_tsquery('vector')
```

---

## Meaning

```text
Matches text search query
```

---

# 24. `ANALYZE`

## Query Planner Statistics

---

## Example

```sql
ANALYZE documents;
```

---

## Important For

* Index usage
* Query optimization

---

# 25. `EXPLAIN ANALYZE`

## Query Performance Debugging

---

## Example

```sql
EXPLAIN ANALYZE
SELECT * FROM documents;
```

---

## Used For

* Performance tuning
* Index debugging

---

# 26. `VACUUM`

## Storage Cleanup

---

## Example

```sql
VACUUM ANALYZE documents;
```

---

# 27. `NOW()`

## Current Timestamp

---

## Example

```sql
created_at >= NOW() - INTERVAL '30 days'
```

---

# 28. `INTERVAL`

## Time Filtering

---

## Example

```sql
INTERVAL '7 days'
```

---

# 29. `JOIN`

## Combine Structured + Vector Data

---

## Example

```sql
SELECT *
FROM embeddings e
JOIN documents d
ON e.document_id = d.id
```

---

# 30. `GROUP BY`

## Aggregation Queries

---

## Example

```sql
GROUP BY department
```

---

# Tier-4: Advanced Enterprise Keywords

---

# 31. `PARTITION BY`

## Large Scale Tables

Used for:

* Billions of embeddings
* Time-based partitioning
* Multi-tenancy

---

# 32. `GIN INDEX`

## JSONB Search Optimization

---

## Example

```sql
CREATE INDEX idx_metadata
ON knowledge_base
USING GIN(metadata);
```

---

# 33. `COALESCE()`

## Null Handling

---

## Example

```sql
COALESCE(score, 0)
```

---

# 34. `DISTINCT`

## Remove Duplicate Results

---

## Example

```sql
SELECT DISTINCT document_id
```

---

# 35. `CASE WHEN`

## Conditional Ranking

---

## Example

```sql
CASE
  WHEN similarity > 0.9 THEN 'High'
  ELSE 'Low'
END
```

---

# Tier-5: Advanced Hybrid Search Symbols

---

# 36. `ts_rank()`

## Full Text Ranking

---

## Example

```sql
ts_rank(to_tsvector(content), plainto_tsquery('ai'))
```

---

# 37. `UNION ALL`

## Merge Multiple Searches

---

## Example

```sql
SELECT ...
UNION ALL
SELECT ...
```

---

# 38. `ROW_NUMBER()`

## Ranking Window Function

---

## Example

```sql
ROW_NUMBER() OVER (
  ORDER BY similarity DESC
)
```

---

# 39. `DESC`

## Descending Sort

---

## Example

```sql
ORDER BY similarity DESC
```

---

# 40. `ASC`

## Ascending Sort

---

## Example

```sql
ORDER BY embedding <=> query_vector ASC
```

---

# Tier-6: Rare but Important

---

# 41. `halfvec`

## Half Precision Vectors

Memory optimized vectors.

---

# 42. `sparsevec`

## Sparse Embeddings

Used in sparse retrieval.

---

# 43. `bit`

## Binary Vectors

Used in compact similarity search.

---

# 44. `LATERAL`

## Advanced Per-Row Vector Search

---

# 45. `MATERIALIZED VIEW`

## Precomputed Search Results

---

# Enterprise Query Example Combining Most Keywords

```sql
SELECT
    id,
    title,
    content,
    metadata,
    1 - (embedding <=> '[0.1,0.2,0.3]') AS similarity,
    
    ts_rank(
        to_tsvector(content),
        plainto_tsquery('pgvector')
    ) AS keyword_rank

FROM knowledge_base

WHERE
    metadata->>'department' = 'AI'
    AND created_at >= NOW() - INTERVAL '30 days'

ORDER BY embedding <=> '[0.1,0.2,0.3]'

LIMIT 10;
```

---

# Most Important Operators Summary

| Operator | Meaning              | Popularity |
| -------- | -------------------- | ---------- |
| `<=>`    | Cosine distance      | ⭐⭐⭐⭐⭐      |
| `<->`    | Euclidean distance   | ⭐⭐⭐⭐       |
| `<#>`    | Inner product        | ⭐⭐⭐⭐       |
| `->>`    | JSON text extraction | ⭐⭐⭐⭐⭐      |
| `@@`     | Full text match      | ⭐⭐⭐⭐       |

---

# Most Important Enterprise Keywords

| Keyword           | Importance     |
| ----------------- | -------------- |
| `ORDER BY`        | Critical       |
| `LIMIT`           | Critical       |
| `WHERE`           | Critical       |
| `USING hnsw`      | Critical       |
| `JSONB`           | Critical       |
| `CREATE INDEX`    | Critical       |
| `EXPLAIN ANALYZE` | Very Important |
| `ANALYZE`         | Very Important |

---

# Most Important Enterprise Concepts

| Concept              | Why Important         |
| -------------------- | --------------------- |
| HNSW                 | Fast ANN search       |
| Metadata filtering   | Enterprise security   |
| Hybrid search        | Better retrieval      |
| Top-K retrieval      | LLM context           |
| Similarity threshold | Reduce hallucinations |
| JSONB filtering      | Multi-tenancy         |
| Full text + vector   | Enterprise search     |
| Query tuning         | Low latency           |

----

# Advanced Enterprise RAG & Retrieval Cheat Sheet

This covers the most important production-grade retrieval concepts used in enterprise GenAI systems with:

* PostgreSQL + pgvector
* FAISS
* Pinecone
* Milvus
* LangChain
* LangGraph

---

# 1. Advanced Hybrid Search Ranking

---

# What is Hybrid Search?

Hybrid search combines:

| Search Type               | Purpose          |
| ------------------------- | ---------------- |
| Keyword search (BM25/FTS) | Exact matching   |
| Semantic search (vectors) | Meaning matching |

---

# Why Hybrid Search Matters

Vector search alone may fail for:

* Exact IDs
* Error codes
* Product names
* API names
* Version numbers

Example:

```text id="zcf5t5"
"KAFKA-1042 timeout issue"
```

Semantic search may miss this exact identifier.

---

# Enterprise Solution

Combine:

```text id="ylt7g7"
BM25 + Vector Search
```

---

# Hybrid Search Architecture

```text id="k7q99p"
User Query
    ↓
Keyword Search (BM25)
    ↓
Vector Search
    ↓
Merge Results
    ↓
Rerank
    ↓
Top Final Results
```

---

# PostgreSQL Hybrid Search Query

```sql id="9lw16t"
SELECT
    id,
    content,
    
    ts_rank(
        to_tsvector(content),
        plainto_tsquery('postgresql vector')
    ) AS keyword_score,

    1 - (embedding <=> '[0.1,0.2,0.3]') AS semantic_score

FROM knowledge_base

WHERE
    to_tsvector(content)
    @@ plainto_tsquery('postgresql vector')

ORDER BY semantic_score DESC
LIMIT 10;
```

---

# Common Hybrid Ranking Formula

```text id="13al7t"
final_score =
(0.7 × semantic_score)
+
(0.3 × keyword_score)
```

---

# Weighted Ranking Formula

FinalScore=\alpha(Semantic)+(1-\alpha)(Keyword)

---

# Enterprise Best Practice

| Data Type   | Better Search |
| ----------- | ------------- |
| PDFs        | Semantic      |
| APIs        | Keyword       |
| Logs        | Keyword       |
| Policies    | Semantic      |
| Source code | Hybrid        |

---

# Production Recommendation

Most enterprise RAG systems use:

```text id="phryfa"
Hybrid Retrieval + Reranking
```

instead of pure vector search.

---

# 2. Reciprocal Rank Fusion (RRF)

---

# What is RRF?

RRF merges multiple ranked result lists.

Used heavily in:

* Enterprise search
* Hybrid search
* Multi-retriever RAG

---

# Why RRF is Powerful

Instead of merging raw scores:

```text id="e7r4kc"
RRF merges rankings
```

which is more stable.

---

# Example

Suppose:

| Document | BM25 Rank | Vector Rank |
| -------- | --------- | ----------- |
| A        | 1         | 5           |
| B        | 2         | 1           |
| C        | 5         | 2           |

RRF combines them intelligently.

---

# RRF Formula

RRF(d)=\sum_{r\in R}\frac{1}{k+r(d)}

---

# Meaning

| Symbol | Meaning            |
| ------ | ------------------ |
| `R`    | Rankers            |
| `r(d)` | Rank position      |
| `k`    | Smoothing constant |

---

# Typical k Value

```text id="v66z8s"
k = 60
```

Most common industry standard.

---

# Why RRF is Better Than Score Averaging

Different retrievers produce:

* Different score scales
* Different distributions

RRF avoids normalization problems.

---

# Enterprise RRF Flow

```text id="dhw1kc"
BM25 Results
        ↓
Vector Results
        ↓
RRF Merge
        ↓
Reranker
        ↓
LLM
```

---

# SQL-Like RRF Example

```sql id="1g95dv"
SELECT *,
(
    1.0 / (60 + keyword_rank)
    +
    1.0 / (60 + vector_rank)
) AS rrf_score

FROM combined_results

ORDER BY rrf_score DESC;
```

---

# Where RRF is Used

| System                | Usage     |
| --------------------- | --------- |
| Search engines        | Very High |
| Enterprise RAG        | Very High |
| Multi-agent retrieval | High      |
| AI copilots           | Very High |

---

# 3. Reranking Pipelines

---

# What is Reranking?

Initial retrieval fetches:

```text id="mjlwm4"
Top 50–200 candidates
```

Then reranker:

```text id="thg8dc"
Reorders results intelligently
```

---

# Why Needed?

Vector retrieval is:

```text id="oypru8"
Approximate
```

Reranker improves precision.

---

# Enterprise Retrieval Pipeline

```text id="r0ot5j"
User Query
    ↓
Hybrid Retrieval
    ↓
Top 100 Documents
    ↓
Cross-Encoder Reranker
    ↓
Top 10 Final Chunks
    ↓
LLM
```

---

# Types of Rerankers

| Type          | Accuracy  | Speed     |
| ------------- | --------- | --------- |
| Cross-encoder | Highest   | Slow      |
| Bi-encoder    | Medium    | Fast      |
| LLM reranker  | Very High | Expensive |

---

# Most Popular Rerankers

| Model                  | Usage        |
| ---------------------- | ------------ |
| BGE Reranker           | Very popular |
| Cohere Rerank          | Enterprise   |
| Jina Reranker          | Modern       |
| Cross-Encoder MS MARCO | Research     |

---

# Cross Encoder Concept

Instead of:

```text id="byvxsi"
Embedding(query)
Embedding(doc)
```

it directly evaluates:

```text id="jz8qtw"
(query + document)
```

together.

---

# Why Cross Encoders Are Better

They understand:

* Context interaction
* Fine-grained relevance
* Exact meaning

---

# Enterprise Best Practice

```text id="o9f7xj"
Retrieve broad → rerank narrow
```

---

# Common Retrieval Sizes

| Stage             | Typical Count |
| ----------------- | ------------- |
| Initial retrieval | 100           |
| After reranking   | 10            |
| Sent to LLM       | 5             |

---

# Reranking Example

---

## Before Rerank

| Rank | Result             |
| ---- | ------------------ |
| 1    | Partially relevant |
| 2    | Exact answer       |
| 3    | Generic content    |

---

## After Rerank

| Rank | Result          |
| ---- | --------------- |
| 1    | Exact answer    |
| 2    | Highly relevant |
| 3    | Related         |

---

# Enterprise Pipeline Example

```python id="0sd8m4"
retrieved_docs = retriever.invoke(query)

reranked_docs = reranker.rerank(
    query,
    retrieved_docs
)

top_docs = reranked_docs[:5]
```

---

# 4. Multi-Vector Retrieval

---

# What is Multi-Vector Retrieval?

Instead of one embedding:

```text id="jlwmf7"
One document → Multiple vectors
```

---

# Why Needed?

One embedding may lose:

* Context
* Structure
* Semantics

---

# Common Multi-Vector Strategies

| Strategy        | Meaning                    |
| --------------- | -------------------------- |
| Chunk vectors   | One vector per chunk       |
| Summary vector  | Document summary embedding |
| Title vector    | Title embedding            |
| Metadata vector | Metadata embedding         |

---

# Architecture

```text id="r4i2f4"
Document
   ↓
Title Embedding
Chunk Embeddings
Summary Embedding
Metadata Embedding
```

---

# Why Enterprises Use This

Improves:

* Recall
* Semantic coverage
* Long-document retrieval

---

# Example

Query:

```text id="jlwm0j"
"refund rules"
```

Maybe:

* Summary vector matches
* Chunk vector misses

---

# Multi-Vector Query Flow

```text id="8m7p2s"
Search title vectors
Search chunk vectors
Search summary vectors
        ↓
Merge results
        ↓
Rerank
```

---

# LangChain MultiVectorRetriever

Widely used production pattern.

---

# Benefits

| Benefit                   | Impact |
| ------------------------- | ------ |
| Better recall             | High   |
| Better long-doc retrieval | High   |
| Better semantic coverage  | High   |

---

# Drawback

| Problem         | Impact |
| --------------- | ------ |
| More storage    | High   |
| More indexing   | High   |
| More complexity | Medium |

---

# 5. Parent-Child Chunk Retrieval

---

# Problem with Normal Chunking

Small chunks:

✅ Better retrieval
❌ Lose context

Large chunks:

✅ More context
❌ Poor retrieval precision

---

# Enterprise Solution

Use:

```text id="3wzwhd"
Child chunks for retrieval
Parent chunks for context
```

---

# Architecture

```text id="72x2jk"
Large Parent Document
        ↓
Small Child Chunks
        ↓
Embed Child Chunks
        ↓
Retrieve Child Chunk
        ↓
Return Parent Context
```

---

# Example

---

## Parent Chunk

```text id="1o6k0m"
Entire policy section
```

---

## Child Chunk

```text id="4tq7do"
Single paragraph
```

---

# Retrieval Flow

```text id="myw5sn"
Query
   ↓
Match child chunk
   ↓
Find parent document
   ↓
Return larger parent context
```

---

# Enterprise Benefits

| Benefit             | Why Important    |
| ------------------- | ---------------- |
| Better precision    | Small chunks     |
| Better context      | Parent retrieval |
| Lower hallucination | More context     |

---

# Most Common Chunk Sizes

| Type         | Tokens    |
| ------------ | --------- |
| Child chunk  | 200–400   |
| Parent chunk | 1000–2000 |

---

# LangChain ParentDocumentRetriever

Very common production approach.

---

# Enterprise Metadata Example

```json id="88z0se"
{
  "parent_id": "DOC_1001",
  "child_id": "DOC_1001_CHUNK_4"
}
```

---

# 6. Query Optimization & Benchmarking

---

# Why Optimization Matters

RAG systems must balance:

| Factor    | Goal |
| --------- | ---- |
| Recall    | High |
| Latency   | Low  |
| Cost      | Low  |
| Precision | High |

---

# Key Enterprise Metrics

---

# 1. Recall@K

Measures:

```text id="2q4w6j"
Did retrieval find relevant documents?
```

---

# Formula

Recall@K=\frac{RelevantRetrieved}{TotalRelevant}

---

# Example

If:

* 10 relevant docs exist
* Retrieved 7

Then:

```text id="hn2e08"
Recall@K = 0.7
```

---

# 2. Precision@K

Measures:

```text id="26dcbm"
How many retrieved docs are correct?
```

---

# Formula

Precision@K=\frac{RelevantRetrieved}{TotalRetrieved}

---

# 3. Latency

Measures:

```text id="1aqy8d"
Query response time
```

---

# Enterprise Targets

| System         | Good Latency |
| -------------- | ------------ |
| Chatbot        | < 2 sec      |
| Enterprise RAG | < 5 sec      |
| Real-time AI   | < 500 ms     |

---

# 4. MRR (Mean Reciprocal Rank)

Measures ranking quality.

---

# Formula

MRR=\frac{1}{|Q|}\sum_{i=1}^{|Q|}\frac{1}{rank_i}

---

# Optimization Techniques

---

# HNSW Tuning

```sql id="1ftk6z"
SET hnsw.ef_search = 100;
```

Higher:

* Better recall
* Slower query

---

# Metadata Prefiltering

```sql id="o8l82d"
WHERE department = 'finance'
```

Reduces search space.

---

# Top-K Optimization

Bad:

```sql id="6g9mjz"
LIMIT 1000
```

Better:

```sql id="cqv8cw"
LIMIT 20
```

---

# Batch Embedding Generation

Instead of:

```text id="qz7rgi"
1 API call per chunk
```

Use batching.

---

# Enterprise Benchmark Pipeline

```text id="qlxwyx"
Dataset
   ↓
Queries
   ↓
Retriever
   ↓
Ground Truth Comparison
   ↓
Metrics
   ↓
Recall / Precision / Latency
```

---

# Enterprise Retrieval Targets

| Metric             | Target  |
| ------------------ | ------- |
| Recall@10          | > 0.85  |
| Precision@5        | > 0.80  |
| Latency            | < 2 sec |
| Hallucination Rate | Low     |

---

# Final Enterprise Retrieval Pipeline

```text id="wfej2s"
User Query
     ↓
Hybrid Retrieval
     ↓
BM25 + Vector Search
     ↓
RRF Merge
     ↓
Reranker
     ↓
Parent Context Expansion
     ↓
Top Final Chunks
     ↓
LLM Generation
```

---

# Production-Grade Retrieval Stack

| Layer         | Technology            |
| ------------- | --------------------- |
| Vector DB     | PostgreSQL + pgvector |
| ANN Search    | HNSW                  |
| Retrieval     | Hybrid                |
| Fusion        | RRF                   |
| Reranking     | Cross Encoder         |
| Orchestration | LangGraph             |
| Framework     | LangChain             |
| Embeddings    | OpenAI / BGE          |
| Cache         | Redis                 |

---